In [1]:
# ! brew install ollama
# ! ollama serve
# ! ollama pull llama3:8b
! ollama list

NAME         ID              SIZE      MODIFIED    
llama3:8b    365c0bd3c000    4.7 GB    3 hours ago    


In [2]:
import json
import pandas as pd
from typing import Dict, List, Any
import numpy as np

def extract_gtfs(file_path):
    json_data = Dict[str, Any]
    with open(file_path, 'r', encoding='utf-8') as f:
        json_data = json.load(f)

    normalized_data = []
    for poi in json_data['elements']:
        # Copy basic properties like type, id, lat, lon
        item = {k: v for k, v in poi.items() if k != 'tags'}
        if 'tags' in poi and isinstance(poi['tags'], dict):
            item.update(poi['tags'])

        normalized_data.append(item)

    df = pd.DataFrame(normalized_data)
    df = df.replace('nan', np.nan)

    if not df.empty:
        priority_cols = ['id', 'name', 'lat', 'lon', 'opening_hours', 'amenity', 'cuisine', 'wheelchair', 'toilets', 'access']
        existing_cols = [col for col in priority_cols if col in df.columns]
        other_cols = [col for col in df.columns if col not in existing_cols]
        df = df[existing_cols + other_cols]

    return df

file_path = './data_collection/raw_data/raw_osm.json'
data_df = extract_gtfs(file_path)

In [3]:
data_df['amenity'].unique()

array([nan, 'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain',
       'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium',
       'conference_centre', 'fire_station', 'cinema', 'toilets',
       'arts_centre', 'place_of_worship'], dtype=object)

## Open model (Llama)

In [4]:
# User-inputted variables
START_LAT = 32.5106
START_LON = -117.0626 
NUM_POIS_TO_VISIT = 4
TIME_PER_POI = 1.5
MAX_DIST = 15

# User Preferences
USER_PREFERNCES = {
    "start_location": {"lat": START_LAT, "lon": START_LON},
    "num_poi": NUM_POIS_TO_VISIT,
    "time_per_poi": TIME_PER_POI,
    "max_travel_dist": MAX_DIST,
    "avg_travel_speed_mph": 20.0, 
    "max_travel_time_minutes": 45.0,
    "amenity_type": "restaurant|theatre|cafe", #'fast_food', 'restaurant', 'cafe', 'post_office', 'fountain', 'bench', 'theatre', 'clock', 'bicycle_parking', 'planetarium', 'conference_centre', 'fire_station', 'cinema', 'toilets', 'arts_centre', 'place_of_worship'
    "tourism_type": "museum|gallery|viewpoint|attraction|aquarium", # 'attraction', 'gallery', 'museum', 'viewpoint', 'artwork', 'aquarium', 'zoo', 'theme_park'
    "cuisine": "italian|mexican|american",
    "required_accessibility": ["wheelchair"], #["wheelchair", "toilets:wheelchair"],
    "visited_ids": set()
}

In [52]:
prompt = f"""
<system>
You are an expert AI itinerary planner for accessible tourism in San Diego.
All responses must be **pure JSON** — no explanations, no markdown, no ellipsis (...), no intro text.
Your output must start with '{{' and end with '}}'. "id" must be the first key.
</system>

You are planning an accessible trip in San Diego for a user with limited mobility.

Here are candidate POIs (filtered by proximity):
{data_df.to_dict(orient='records')}

User preferences:
{USER_PREFERNCES}

### Optimization Objective
You must choose **exactly 1 POI** that best satisfies the following weighted scoring function:

Score = (AccessibilityScore * 0.5) + (DistanceScore * 0.3) + (PreferenceMatch * 0.2)

- AccessibilityScore = 1.0 if "wheelchair" == True, else 0.0.
- DistanceScore = 1.0 if within 5 km of ({START_LAT}, {START_LON}), 0.5 if 5–10 km, else 0.
- PreferenceMatch = 1.0 if POI matches "amenity_type" or "cuisine" in user preferences, else 0.5 if partially matches.

Think step-by-step internally to rank all POIs by this score.
The chosen POI must from the candidate list.
Then output **only the best one** in the following JSON format:

[
  {{
    "id": int,
    "name": str,
    "lat": float,
    "lon": float,
    "poi_type": "amenity" | "tourism",
    "features": {{
      "wheelchair": bool,
      "toilets:wheelchair": bool,
      "air_conditioning": bool
    }},
    "amenity" or "tourism": str,
    "cuisine": str
  }}
]

The "id" field must be the first key in the output JSON object.
The example output:
{{
  "id": 123456,
  "name": "Example POI",
  "lat": 32.7157,
  "lon": -117.1611,
  "poi_type": "amenity",
  "features": {{
    "wheelchair": true,
    "toilets:wheelchair": false,
    "air_conditioning": true
  }},
  "amenity": "restaurant",
  "cuisine": "italian"
}}

### Important:
- Do not output reasoning.
- Output only valid JSON, fully written out, no ellipsis.
"""


In [ ]:
import subprocess
import json
import re


def run_ollama(prompt, model="llama3:8b"):
    result = subprocess.run(
        ["ollama", "run", model, "--format", "json"],
        input=prompt,
        text=True,
        capture_output=True
    )
    return result.stdout

itinerary = []
generated_names = set()

for i in range(NUM_POIS_TO_VISIT):
    while True:
        response = run_ollama(prompt)
        raw_text = response.strip()

        response_dict = re.sub(r'\btrue\b', 'True', raw_text, flags=re.IGNORECASE)
        response_dict = re.sub(r'\bfalse\b', 'False', response_dict, flags=re.IGNORECASE)
        response_dict = re.sub(r'\bnull\b', 'None', response_dict, flags=re.IGNORECASE)

        try:
            response_dict = eval(response_dict)
        except Exception as e:
            continue

        name = response_dict.get("name")
        if not name:
            continue

        if name in generated_names:
            print(f"⚠️ '{name}' exist, regenerating...")
            continue
        else:
            generated_names.add(name)
            itinerary.append(response_dict)
            print(f"✅ Added POI: {name}")
            break

# for i in range(NUM_POIS_TO_VISIT):
#     response = run_ollama(prompt)
#     raw_text = response.strip()
#     response_dict = re.sub(r'\btrue\b', 'True', raw_text, flags=re.IGNORECASE)
#     response_dict = re.sub(r'\bfalse\b', 'False', response_dict, flags=re.IGNORECASE)
#     response_dict = re.sub(r'\bnull\b', 'None', response_dict, flags=re.IGNORECASE)
#     response_dict = eval(response_dict)
#     print("Response:", response_dict)
#     itinerary.append(response_dict)
# print(itinerary)


✅ Added POI: San Diego Museum of Man
⚠️ 'San Diego Museum of Man' exist, regenerating...
⚠️ 'San Diego Museum of Man' exist, regenerating...
✅ Added POI: Old Town San Diego State Historic Park
⚠️ 'Old Town San Diego State Historic Park' exist, regenerating...
✅ Added POI: Old Town San Diego
⚠️ 'San Diego Museum of Man' exist, regenerating...
✅ Added POI: Old Town Trolley Tours San Diego


In [70]:
print(itinerary)

[{'id': 1345675, 'name': 'San Diego Museum of Man', 'lat': 32.7543105, 'lon': -117.1968079, 'poi_type': 'tourism', 'features': {'wheelchair': True}, 'amenity': 'museum', 'cuisine': 'none'}, {'id': '123456', 'name': 'Old Town San Diego State Historic Park', 'lat': 32.7122, 'lon': -117.1633, 'poi_type': 'tourism', 'features': {'wheelchair': True, 'toilets:wheelchair': False, 'air_conditioning': False}, 'amenity': 'park', 'cuisine': ''}, {'id': 123456, 'name': 'Old Town San Diego', 'lat': 32.7123, 'lon': -117.1632, 'poi_type': 'tourism', 'features': {'wheelchair': True}, 'amenity': 'museum', 'cuisine': 'american'}, {'id': -1, 'name': 'Old Town Trolley Tours San Diego', 'lat': 32.7239, 'lon': -117.1668, 'poi_type': 'tourism', 'features': {'wheelchair': True}, 'amenity': 'attraction', 'cuisine': ''}]


In [69]:
from evaluation import get_evaluation_metrics_verbose

metrics = get_evaluation_metrics_verbose(itinerary, USER_PREFERNCES)


--------------------------------------------------------------------------------
Itinerary
--------------------------------------------------------------------------------
Total POIs: 4
User Preferences: {'start_location': {'lat': 32.5106, 'lon': -117.0626}, 'num_poi': 4, 'time_per_poi': 1.5, 'max_travel_dist': 15, 'avg_travel_speed_mph': 20.0, 'max_travel_time_minutes': 45.0, 'amenity_type': 'restaurant|theatre|cafe', 'tourism_type': 'museum|gallery|viewpoint|attraction|aquarium', 'cuisine': 'italian|mexican|american', 'required_accessibility': ['wheelchair'], 'visited_ids': set()}
Required Features: ['wheelchair']

--------------------------------------------------------------------------------
Quality of POI Metric
--------------------------------------------------------------------------------
Total Travel Distance: 22.90 miles
Total Travel Time: 68.7 minutes (Max: 45.0 min)
Travel Distance/Time Score: -0.53
POI Diversity Score: 0.25
Preference Coverage: 0.00

--------------------